In [1]:
import pandas as pd
# import matplotlib.pyplot as plt
# from matplotlib.ticker import FuncFormatter
from toolsos.huisstijl.colors import get_os_colors
# from graphs import simple_barh, simple_line, multiple_line
import pickle
from itertools import cycle

In [ ]:
zo = pd.read_excel('../../04 tabellen/01 tabellen zuidoost/tabel alle data long Masterplan Zuidoost 2026_05.xlsx')
nw = pd.read_excel('../../04 tabellen/03 tabellen nieuwwest/tabel alle data long Samen Nieuw-West 2026_05.xlsx')
no = pd.read_excel('../../04 tabellen/02 tabellen noord/tabel alle data long Aanpak Noord 2026_05.xlsx')

zo = pd.read_csv('../../04 tabellen/')

In [13]:
# amsterdam cijfers verwijderen in 2 vd 3 datasets anders 3x dubbel na concat
no = no[no['spatial_type']!= 'gemeente']
nw = nw[nw['spatial_type']!= 'gemeente']

In [14]:
df_list = [zo, nw, no]

df_onbewerkt = pd.concat(df_list)
df_onbewerkt = df_onbewerkt[
    [
        "indicator_sd",
        "measure",
        "value",
        "temporal_date",
        "spatial_type",
        "spatial_name",
        "spatial_code",
        "kernindicator_noord",
        "kernindicator_zo",
        "kernindicator_nw",
        "tweedeling_def",
    ]
]
df_onbewerkt.drop_duplicates(inplace=True)

In [1]:
def bewerk_dataframe(df_onbewerkt):
    df = df_onbewerkt.loc[
        # (
        #     # (df_onbewerkt["measure"].isin(indicatoren)) &
        #     (df_onbewerkt["kernindicator_zo"] == 1)
        #     | (df_onbewerkt["kernindicator_nw"] == "primair")
        #     | (df_onbewerkt["kernindicator_noord"] == 1)
        # ) &
        (df_onbewerkt["spatial_type"].isin(["stadsdelen", "gemeente"]))
        (df_onbewerkt["tweedeling_def"] == "totaal")
    ].copy()

    df["temporal_date"] = pd.to_datetime(df["temporal_date"], format="%Y")
    df["temporal_date"] = df["temporal_date"].dt.year.astype(str)

    df = df.sort_values(["spatial_name", "measure", "temporal_date"])

    return df


df = bewerk_dataframe(df_onbewerkt)

NameError: name 'df_onbewerkt' is not defined

In [18]:
# 1 grote dataset maken in vorm van dictionary, met als key de variabele.
indicatoren  = df['measure'].unique()

resultaten_sd = {}

for var in indicatoren:
    resultaten_sd[f"{var}_zo"] = df[(df["measure"] == var) & (df["spatial_name"] == "Zuidoost")].copy()
    resultaten_sd[f"{var}_nw"] = df[(df["measure"] == var) & (df["spatial_name"] == "Nieuw-West")].copy()
    resultaten_sd[f"{var}_no"] = df[(df["measure"] == var) & (df["spatial_name"] == "Noord")].copy()

resultaten_ams = {}

for var in indicatoren:
    resultaten_ams[f"{var}_zo"] = df[(df["measure"] == var) & (df["spatial_name"].isin(["Amsterdam","Zuidoost"]))].copy()
    resultaten_ams[f"{var}_nw"] = df[(df["measure"] == var) & (df["spatial_name"].isin(["Amsterdam","Nieuw-West"]))].copy()
    resultaten_ams[f"{var}_no"] = df[(df["measure"] == var) & (df["spatial_name"].isin(["Amsterdam","Noord"]))].copy()

resultaten = {}

for var in indicatoren:
    resultaten[f"{var}"] = df[(df["measure"] == var)].copy()

In [20]:
# Opslaan als excel
df.to_excel('../../03 tussentijds/data figuren/resultaten.xlsx')

# Opslaan als dictionary met dfs

out_path = '../../03 tussentijds/data figuren'

with open(f"{out_path}/resultaten_ams.pkl", "wb") as f:
    pickle.dump(resultaten_ams, f)

with open(f"{out_path}/resultaten.pkl", "wb") as f:
    pickle.dump(resultaten, f)

with open(f"{out_path}/resultaten_sd.pkl", "wb") as f:
    pickle.dump(resultaten_sd, f)